# LLM 社会模拟预实验 - Jupyter Notebook

本 Notebook 用于运行 LLM 社会模拟预实验。

**使用方法**：从上到下依次点击每个代码单元格左侧的 ▶ 运行按钮。

## 第 1 步：检查并安装依赖

In [ ]:
import importlib
import sys
import subprocess

required = {
    "openai": "openai",
    "yaml": "PyYAML",
    "networkx": "networkx",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}

missing = []
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        missing.append(package)

if missing:
    print(f"正在安装缺失的依赖: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("安装完成！")
else:
    print("所有依赖已安装 ✓")

## 第 2 步：设置 API Key

在下方输入框填入你的阿里云百炼 API Key，然后运行。

In [ ]:
import os

# 从环境变量读取（如果已配置）
api_key = os.environ.get("DASHSCOPE_API_KEY", "")

# 如果没有设置，在这里手动输入
if not api_key:
    api_key = input("请输入你的阿里云百炼 API Key: ").strip()

if api_key:
    os.environ["DASHSCOPE_API_KEY"] = api_key
    print("API Key 已设置 ✓")
else:
    print("⚠️ 未设置 API Key，请在输入框中填写")

## 第 3 步：导入实验模块

In [ ]:
import sys
from pathlib import Path

# 将 src 目录加入路径
src_path = Path("src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from llm_client import LLMClient, extract_score
from network import build_ba_network, network_stats, sample_neighbors, build_neighbor_context
from agent import Agent, default_roles, build_messages
from metrics import round_metrics, convergence_half_life

print("模块导入成功 ✓")

## 第 4 步：配置实验参数

可以修改下面的参数来配置实验。

In [ ]:
# 实验配置
CONFIG = {
    "topic": "AI取代劳动力",
    "seed": 42,
    "experiment": {
        "n_agents": 20,       # Agent 数量
        "n_rounds": 30,       # 轮数（增加轮数观察长期趋势）
        "n_activated_per_round": 8,  # 每轮激活数量（约 40%）
    },
    "network": {
        "type": "ba",
        "m": 2,              # BA 网络参数
    },
    "sampling": {
        "alpha": 0.5,        # 邻居采样加权系数
        "max_neighbors": 10, # 最多采样邻居数
        "memory_window": 3,  # 记忆窗口
    },
    "generation": {
        "temperature": 0.6,   # 降低随机性，Agent 更稳定
        "max_tokens": 120,
        "concurrency": 3,     # API 并发数
        "max_retries": 2,
        "timeout_seconds": 60,
    },
    "model": {
        "provider": "qwen_api",
        "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
        "model": "qwen3-8b",
        "api_key_env": "DASHSCOPE_API_KEY",
    }
}

print("配置完成 ✓")
print(f"话题: {CONFIG['topic']}")
print(f"Agent 数: {CONFIG['experiment']['n_agents']}")
print(f"轮数: {CONFIG['experiment']['n_rounds']}")
print(f"模型: {CONFIG['model']['model']}")
print(f"预计 API 调用次数: {CONFIG['experiment']['n_agents'] * CONFIG['experiment']['n_rounds']} 次")

## 第 5 步：构建网络

In [ ]:
n_agents = CONFIG["experiment"]["n_agents"]
m = CONFIG["network"]["m"]
seed = CONFIG["seed"]

graph, agents = build_ba_network(n_agents=n_agents, m=m, seed=seed)

print("网络构建完成 ✓")
print(f"节点数: {graph.number_of_nodes()}")
print(f"边数: {graph.number_of_edges()}")
print(f"网络统计: {network_stats(graph)}")
print("\nAgent 初始状态:")
for aid, agent in agents.items():
    stubborn_label = "固执" if agent.stubbornness > 0.5 else "灵活"
    print(f"  Agent {aid}: {agent.role_description}")
    print(f"    初始评分 {agent.initial_score}/10, 固执度 {agent.stubbornness} ({stubborn_label})")

## 第 6 步：创建 LLM 客户端

In [ ]:
model_config = CONFIG["model"]

client = LLMClient(
    base_url=model_config["base_url"],
    model=model_config["model"],
    api_key_env=model_config["api_key_env"],
    max_retries=CONFIG["generation"]["max_retries"],
    timeout_seconds=CONFIG["generation"]["timeout_seconds"],
)

print("LLM 客户端创建成功 ✓")
print(f"使用模型: {model_config['model']}")

## 第 7 步：运行模拟

⚠️ **注意**：这会调用 API 产生费用。20 agents × 30 轮 = 约 600 次 API 调用。

In [ ]:
import asyncio
import random
import json
import csv
from datetime import datetime

# 实验参数
topic = CONFIG["topic"]
rng = random.Random(CONFIG["seed"])

exp_cfg = CONFIG["experiment"]
sampling_cfg = CONFIG["sampling"]
gen_cfg = CONFIG["generation"]

n_rounds = exp_cfg["n_rounds"]
n_activated = exp_cfg["n_activated_per_round"]
alpha = sampling_cfg["alpha"]
max_neighbors = sampling_cfg["max_neighbors"]
memory_window = sampling_cfg["memory_window"]
temperature = gen_cfg["temperature"]
max_tokens = gen_cfg["max_tokens"]
concurrency = gen_cfg["concurrency"]

# 结果存储
all_outputs = []
all_round_metrics = []
all_nodes = list(graph.nodes())

# 创建结果目录
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = Path("results/pilot") / timestamp
results_dir.mkdir(parents=True, exist_ok=True)
print(f"结果保存目录: {results_dir}")

# Semaphore 控制并发
semaphore = asyncio.Semaphore(concurrency)

async def process_agent(round_id, node_id):
    sampled = sample_neighbors(graph, node_id, alpha, max_neighbors, rng)
    neighbor_context = build_neighbor_context(agents, sampled, memory_window)
    messages = build_messages(agents[node_id], topic, neighbor_context)
    
    async with semaphore:
        raw_response = await asyncio.to_thread(
            client.generate,
            messages,
            temperature,
            max_tokens,
        )
    
    parsed_score = extract_score(raw_response)
    parse_ok = parsed_score is not None
    final_score = parsed_score if parsed_score is not None else agents[node_id].current_score
    
    return {
        "round": round_id,
        "agent_id": node_id,
        "role_description": agents[node_id].role_description,
        "score": final_score,
        "parse_ok": parse_ok,
        "text": raw_response.strip(),
    }

# 运行每轮
for round_id in range(1, n_rounds + 1):
    print(f"\n{'='*50}")
    print(f"开始第 {round_id}/{n_rounds} 轮...")
    
    # 选择激活节点
    if n_activated >= len(all_nodes):
        active_nodes = all_nodes[:]
    else:
        active_nodes = sorted(rng.sample(all_nodes, n_activated))
    
    # 并发生成
    outputs = await asyncio.gather(
        *(process_agent(round_id, node_id) for node_id in active_nodes)
    )
    
    # 统一写入
    for row in outputs:
        agents[row["agent_id"]].add_record(
            round_id=round_id,
            text=row["text"],
            score=int(row["score"]),
        )
        graph.nodes[row["agent_id"]]["current_score"] = int(row["score"])
        all_outputs.append(row)
    
    # 计算指标
    current_scores = [agent.current_score for agent in agents.values()]
    parse_ok_values = [row["parse_ok"] for row in outputs]
    metrics_row = round_metrics(round_id, current_scores, parse_ok_values)
    all_round_metrics.append(metrics_row)
    
    print(f"第 {round_id} 轮完成")
    print(f"  平均评分: {metrics_row['mean_score']:.2f}")
    print(f"  评分方差: {metrics_row['variance_score']:.2f}")
    print(f"  最低/最高: {metrics_row['min_score']} / {metrics_row['max_score']}")
    print(f"  解析失败率: {metrics_row['parse_failure_rate']:.0%}")

print(f"\n{'='*50}")
print("模拟完成！")

## 第 8 步：保存结果

In [ ]:
# 保存原始输出
raw_path = results_dir / "raw_outputs.jsonl"
with raw_path.open("w", encoding="utf-8") as f:
    for row in all_outputs:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

# 保存指标
metrics_path = results_dir / "round_metrics.csv"
if all_round_metrics:
    with metrics_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(all_round_metrics[0].keys()))
        writer.writeheader()
        writer.writerows(all_round_metrics)

# 保存配置
import yaml
with (results_dir / "config_used.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)

# 保存网络统计
with (results_dir / "network_stats.json").open("w", encoding="utf-8") as f:
    json.dump(network_stats(graph), f, ensure_ascii=False, indent=2)

# 保存摘要
summary = {
    "run_dir": str(results_dir),
    "network_stats": network_stats(graph),
    "convergence_half_life": convergence_half_life(all_round_metrics),
    "final_metrics": all_round_metrics[-1] if all_round_metrics else None,
}
with (results_dir / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"结果已保存到: {results_dir}")
print(f"\n文件列表:")
for f in results_dir.iterdir():
    print(f"  {f.name}")

## 第 9 步：查看结果

In [ ]:
import pandas as pd

# 读取指标
df = pd.DataFrame(all_round_metrics)
print("每轮指标:")
print(df.to_string(index=False))

# 显示每个 agent 的输出
print("\n\nAgent 输出详情:")
for row in all_outputs:
    print(f"\n[Round {row['round']} | Agent {row['agent_id']}]")
    print(f"角色: {row['role_description']}")
    print(f"评分: {row['score']}")
    print(f"内容: {row['text'][:100]}...")

## 第 10 步：绘制图表

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

axes[0].plot(df["round"], df["mean_score"], marker="o")
axes[0].set_ylabel("平均评分")
axes[0].set_ylim(1, 10)
axes[0].grid(True, alpha=0.3)
axes[0].set_title("观点评分动态变化")

axes[1].plot(df["round"], df["variance_score"], marker="o", color="tab:orange")
axes[1].set_xlabel("轮次")
axes[1].set_ylabel("方差")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()

# 保存图片
fig_path = results_dir / "score_dynamics.png"
fig.savefig(fig_path, dpi=200)
print(f"图表已保存: {fig_path}")

plt.show()

---

## 后续操作

如果上面运行成功，你可以：

1. **修改参数重新运行**：回到第 4 步，修改 `CONFIG` 中的参数，再依次运行后续步骤
2. **扩大规模**：把 `n_agents` 改为 50，`n_rounds` 改为 50
3. **切换模型**：修改 `CONFIG['model']['model']` 为其他模型（如 qwen-plus）

## 参数调优建议

| 现象 | 可能原因 | 调整方案 |
|------|---------|---------|
| 方差下降太慢 | Agent 不够灵活 | 提高 temperature 到 0.9 |
| 评分解析失败率高 | 模型输出格式不稳定 | 换 qwen-plus 或简化 prompt |
| 观点完全不收敛 | 初始立场太分散 | 减小初始评分方差 |
| 收敛太快 | 温度太高或 Agent 太容易被影响 | 降低 temperature 到 0.6 |

## 常见问题

- **API 报错**：检查 API Key 是否正确，或网络是否通畅
- **评分解析失败**：看 `parse_fail` 是否过高，如果高可能需要调整 prompt 或换模型
- **图表不显示**：确保安装了 matplotlib
- **运行太慢**：减小 `concurrency`（并发数）避免 API 限流